# CSDI-adapted imputation of the full dataset (downstream benchmark)

This notebook imputes the full dataset with the same two-tier rule as CAKGI:

| Gap length | Method |
|---|---|
| < 30 min | linear interpolation |
| 30 min – 8 h | CSDI-adapted |
| > 8 h | left missing |

CSDI-adapted uses the same adaptive window as CAKGI, min(120 + 0.5·duration, 360) min on each side. It is initialised from noise, runs the full T = 100 reverse steps, and outputs the median of 50 chains with 10th–90th percentile intervals.

**Input:** `SDBII.csv`.
**Output:** `SDBII_imputed_csdi.csv`, with the same columns as the BRITS and CAKGI outputs; imputed values are labelled `csdi_long`.

#  Imports + Constants

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os, gc, warnings
warnings.filterwarnings('ignore')

BASE             = r"path/to/your/data_directory"     # <-- set this
ORIGINAL_FILE    = f"{BASE}/SDBII.csv"
MODEL_FILE       = f"{BASE}/SDBII_csdi_model.pt"
FINAL_FILE       = f"{BASE}/SDBII_imputed_csdi.csv"
CHECKPOINT_FILE  = f"{BASE}/SDBII_imputed_csdi_ckpt.csv"

FREQ       = '5min'
CTX_WIN    = 120     # context window for diffusion methods
MAX_L      = 1500    # max window timesteps
T_STEPS    = 100
N_SAMPLES  = 50      # chains — matches PIDCK for fair comparison
                     # MC eval also uses N=50
SEQ_LEN    = 48
MASK_RATIO = 0.15
EPOCHS     = 75
BATCH_SIZE = 32
LR         = 1e-3
NF         = 2       # glucose + HR
HYPO_THR      = 70
MAX_GAP_MINS  = 480    # 8 hours — max imputable gap (matches MC eval)

SEED = 2026
np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Method: csdi | Device: {DEVICE}")


#  Load Data + Gap Classification

In [ ]:
df = pd.read_csv(ORIGINAL_FILE, sep=None, engine='python')
df['ts'] = pd.to_datetime(df['ts'], format='mixed', dayfirst=True)
df = df.sort_values(['PtID','ts']).reset_index(drop=True)
df['GlucoseCGM'] = df['GlucoseCGM'].replace(0, np.nan)
df['HR']         = df['HR'].replace(0, np.nan)


G_MEAN = df['GlucoseCGM'].mean(); G_STD = df['GlucoseCGM'].std()
H_MEAN = df['HR'].mean();          H_STD = df['HR'].std()
G_STD  = G_STD if G_STD > 0 else 1.
H_STD  = H_STD if H_STD > 0 else 1.



def classify_gaps(pat_df, col, freq='5min', thr=30):
    """
    Labels EVERY missing timestep in each gap:
      'small'   — gap < 30 min  -> linear interpolation
      'long'    — 30 min <= gap <= MAX_GAP_MINS (8hr) -> method imputation
      'toolong' — gap > 8hr -> left as missing (no method can reliably impute)
    """
    full = pd.date_range(pat_df['ts'].min(), pat_df['ts'].max(), freq=freq)
    s    = pat_df.set_index('ts')[col].reindex(full)
    mis  = s.isna()
    lbl  = pd.Series(None, index=full, dtype=object)
    if mis.any():
        gid = (mis != mis.shift()).cumsum()[mis]
        for g, grp in s[mis].groupby(gid):
            dur_mins = len(grp) * 5
            if dur_mins < thr:
                gap_type = 'small'
            elif dur_mins <= MAX_GAP_MINS:
                gap_type = 'long'
            else:
                gap_type = 'toolong'   # > 8hr: skip imputation
            lbl.loc[grp.index] = gap_type
    result = pat_df['ts'].map(lbl[lbl.notna()].to_dict())
    result.index = pat_df.index
    return result

df['GlucoseCGM_gap'] = None; df['HR_gap'] = None
for pid, g in df.groupby('PtID'):
    df.loc[g.index,'GlucoseCGM_gap'] = classify_gaps(g,'GlucoseCGM').values
    df.loc[g.index,'HR_gap']         = classify_gaps(g,'HR').values

print(f"Loaded {len(df):,} rows | {df['PtID'].nunique()} patients")
print("Glucose gaps:", df['GlucoseCGM_gap'].value_counts(dropna=True).to_dict())
print("HR gaps:",      df['HR_gap'].value_counts(dropna=True).to_dict())


#  Real CSDI Model

In [ ]:
METHOD = 'csdi'

D_TIME_EMB=128; CSDI_DM=64; CSDI_NH=4; CSDI_NL=4

class DiffEmb(nn.Module):
    def __init__(self,d_time=D_TIME_EMB,d_out=CSDI_DM):
        super().__init__()
        self.d=d_time; self.p1=nn.Linear(d_time,d_out); self.p2=nn.Linear(d_out,d_out)
    def forward(self,t):
        h=self.d//2; f=torch.exp(-np.log(10000)*torch.arange(h,device=t.device)/h)
        a=t[:,None].float()*f[None]; e=torch.cat([a.sin(),a.cos()],dim=-1)
        return torch.nn.functional.silu(self.p2(torch.nn.functional.silu(self.p1(e))))

class CSDIResBlock(nn.Module):
    def __init__(self,dm=CSDI_DM,nh=CSDI_NH,nf=NF):
        super().__init__()
        self.t_proj=nn.Linear(dm,dm)
        self.conv1=nn.Conv1d(dm,dm*2,kernel_size=3,padding=1)
        self.t_attn=nn.MultiheadAttention(dm,nh,batch_first=True)
        self.t_ln=nn.LayerNorm(dm)
        self.f_fc=nn.Sequential(nn.Linear(dm,dm*2),nn.GELU(),nn.Linear(dm*2,dm))
        self.f_ln=nn.LayerNorm(dm)
        self.out_proj=nn.Conv1d(dm,dm,kernel_size=1)
        self.skip_proj=nn.Conv1d(dm,dm,kernel_size=1)
    def forward(self,x,t_emb):
        h=x+self.t_proj(t_emb).unsqueeze(-1)
        h=self.conv1(h); h1,h2=h.chunk(2,dim=1)
        h=torch.tanh(h1)*torch.sigmoid(h2)
        ht=h.permute(0,2,1); ht_a,_=self.t_attn(ht,ht,ht)
        ht=self.t_ln(ht+ht_a)
        hf=self.f_ln(ht+self.f_fc(ht)).permute(0,2,1)
        return (self.out_proj(hf)+x)/np.sqrt(2),self.skip_proj(hf)

class CSDIModel(nn.Module):
    def __init__(self,nf=NF,dm=CSDI_DM,nh=CSDI_NH,nl=CSDI_NL,T=T_STEPS):
        super().__init__()
        self.T=T; self.nl=nl
        self.input_proj=nn.Conv1d(nf*3,dm,kernel_size=1)
        self.t_emb=DiffEmb(D_TIME_EMB,dm)
        self.layers=nn.ModuleList([CSDIResBlock(dm,nh,nf) for _ in range(nl)])
        self.output_proj=nn.Sequential(
            nn.Conv1d(dm,dm,kernel_size=1),nn.ReLU(),nn.Conv1d(dm,nf,kernel_size=1))
        b=self._cs(T); self.register_buffer('alpha_bar',torch.cumprod(1-b,0))
    def _cs(self,T,s=0.008):
        st=torch.arange(T+1,dtype=torch.float)
        f=torch.cos(((st/T+s)/(1+s))*np.pi/2)**2; f=f/f[0]
        return torch.clamp(1-f[1:]/f[:-1],0.0001,0.9999)
    def forward(self,x_noisy,t,x_observed,mask):
        inp=torch.cat([x_noisy,x_observed,mask],dim=1)
        h=self.input_proj(inp); te=self.t_emb(t); skip=torch.zeros_like(h)
        for layer in self.layers: h,s=layer(h,te); skip+=s
        return self.output_proj(skip/np.sqrt(self.nl))
    def loss(self,x0,mask):
        # (B,NF,L) format for Conv1d
        B=x0.shape[0]; t=torch.randint(1,self.T+1,(B,),device=x0.device)
        ab=self.alpha_bar[t-1].view(B,1,1); eps=torch.randn_like(x0)
        xn=ab.sqrt()*x0+(1-ab).sqrt()*eps
        pred=self.forward(xn*(1-mask),t,x0*mask,mask)
        miss=1-mask
        return torch.nn.functional.mse_loss(pred*miss,eps*miss) if miss.sum()>0 else torch.nn.functional.mse_loss(pred*mask,eps*mask)

class CSDIDataset(Dataset):
    def __init__(self,df,seq=SEQ_LEN,mr=MASK_RATIO):
        self.s=[]
        for pid,p in df.groupby('PtID'):
            p=p.sort_values('ts').reset_index(drop=True)
            g=(p['GlucoseCGM'].values-G_MEAN)/G_STD
            h=(p['HR'].values-H_MEAN)/H_STD
            sig=np.stack([g,h],axis=0)  # (NF,T)
            for i in range(0,sig.shape[1]-seq,seq//2):
                w=sig[:,i:i+seq]
                if np.isnan(w).any(): continue
                self.s.append(w)
    def __len__(self): return len(self.s)
    def __getitem__(self,i):
        x=torch.tensor(self.s[i],dtype=torch.float32)
        m=torch.ones_like(x)
        nm=max(1,int(SEQ_LEN*MASK_RATIO))
        st=int(torch.randint(0,SEQ_LEN-nm+1,(1,)).item())
        m[:,st:st+nm]=0
        return x,m

model=CSDIModel().to(DEVICE)
print(f"CSDI: {sum(p.numel() for p in model.parameters()):,} params")

if os.path.exists(MODEL_FILE):
    ckpt=torch.load(MODEL_FILE,map_location=DEVICE)
    if 'input_proj.weight' in ckpt:
        try:
            model.load_state_dict(ckpt); print("Model loaded")
        except RuntimeError as e:
            print(f"Checkpoint key mismatch — retraining: {e}")
            os.remove(MODEL_FILE)
    else:
        print("Stale checkpoint — retraining"); os.remove(MODEL_FILE)

if not os.path.exists(MODEL_FILE):
    ds=CSDIDataset(df); ld=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=True)
    opt=torch.optim.Adam(model.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,EPOCHS)
    model.train()
    for ep in range(EPOCHS):
        el=0
        for x,m in ld:
            opt.zero_grad(); l=model.loss(x.to(DEVICE),m.to(DEVICE))
            l.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step(); el+=l.item()
        sch.step()
        if (ep+1)%10==0: print(f"  Ep{ep+1}/{EPOCHS} loss={el/len(ld):.4f}")
    torch.save(model.state_dict(),MODEL_FILE)
    print(f"Model saved: {MODEL_FILE}")


# Short Gap Imputation (linear, < 30 min)

In [ ]:
def impute_short_gaps(df, freq=FREQ):
    df_out = df.copy()
    short_records = []   # track filled positions for label assignment

    for pid, pat in df_out.groupby('PtID'):
        pat_sorted = pat.sort_values('ts')
        for col, gap_col in [('GlucoseCGM','GlucoseCGM_gap'),('HR','HR_gap')]:
            # Find positions labelled 'small'
            small_mask = pat_sorted[gap_col] == 'small'
            if not small_mask.any(): continue

            # Interpolate the full series then keep only 'small' gap positions
            s  = pat_sorted.set_index('ts')[col]
            si = s.interpolate(method='linear', limit_area='inside')

            # Only fill where: (a) originally NaN, (b) gap_col == 'small'
            for idx in pat_sorted[small_mask].index:
                ts_val = pat_sorted.loc[idx, 'ts']
                if pd.isna(pat_sorted.loc[idx, col]):
                    filled_val = si.get(ts_val, np.nan)
                    if not pd.isna(filled_val):
                        df_out.loc[idx, col] = filled_val
                        short_records.append({
                            'PtID': pid, 'ts': ts_val, 'col': col,
                            'value': round(float(filled_val), 3)
                        })

    short_df = pd.DataFrame(short_records) if short_records else pd.DataFrame(
        columns=['PtID','ts','col','value'])
    n_g = (short_df['col']=='GlucoseCGM').sum() if len(short_df) else 0
    n_h = (short_df['col']=='HR').sum() if len(short_df) else 0
    print(f"Short gaps filled: Glucose={n_g:,} | HR={n_h:,} timesteps")
    return df_out.reset_index(drop=True), short_df

df_short, short_df = impute_short_gaps(df)


# Impute Long Gaps with CSDI (N=50, median)

In [ ]:
from tqdm import tqdm

def adaptive_win_csdi(dur):
    win=min(CTX_WIN+dur*0.5,360)
    if (win*2+dur)/5>MAX_L-50: win=max(CTX_WIN,(MAX_L-50-dur/5)*5/2)
    win=max(win,CTX_WIN)
    return round(win/5)*5

model.eval(); long_rows=[]
for pid,pat in tqdm(df_short.groupby('PtID'), desc='CSDI'):
    pat=pat.sort_values('ts').reset_index(drop=True)
    for col,gap_col in [('GlucoseCGM','GlucoseCGM_gap'),('HR','HR_gap')]:
        gid=(pat[gap_col]=='long')
        gid=(gid!=gid.shift()).cumsum()[gid]
        for g,grp in pat[pat[gap_col]=='long'].groupby(gid):
            gs=grp['ts'].iloc[0]; ge=grp['ts'].iloc[-1]
            dur=(ge-gs).total_seconds()/60+5
            wm=adaptive_win_csdi(dur)
            wm=round(wm/5)*5  
            ws=gs-pd.Timedelta(minutes=wm); we=ge+pd.Timedelta(minutes=wm)
            fidx=pd.date_range(ws,we,freq=FREQ); L=len(fidx)
            if L>MAX_L: continue
            seg=pat[(pat['ts']>=ws)&(pat['ts']<=we)]
            gn=(seg.set_index('ts')['GlucoseCGM'].reindex(fidx).values-G_MEAN)/G_STD
            hn=(seg.set_index('ts')['HR'].reindex(fidx).values-H_MEAN)/H_STD
            sig=np.stack([gn,hn],axis=0).astype(np.float32)  # (NF,L)
            msk=(~np.isnan(sig)).astype(np.float32)
            sig=np.nan_to_num(sig,nan=0.)
            xi=torch.tensor(sig,dtype=torch.float32).unsqueeze(0).to(DEVICE)
            mt=torch.tensor(msk,dtype=torch.float32).unsqueeze(0).to(DEVICE)
            gidx=np.where((fidx>=gs)&(fidx<=ge))[0]
            samps=[]
            with torch.no_grad():
                for _ in range(N_SAMPLES):
                    x=torch.randn(1,NF,L,device=DEVICE); x=x*(1-mt)+xi*mt
                    for t in reversed(range(1,model.T+1)):
                        tt=torch.full((1,),t,device=DEVICE,dtype=torch.long)
                        eps=model(x*(1-mt),tt,xi*mt,mt)
                        ab=model.alpha_bar[t-1]
                        abp=model.alpha_bar[t-2] if t>1 else torch.tensor(1.,device=DEVICE)
                        x0=((x-(1-ab).sqrt()*eps)/ab.sqrt()).clamp(-3,3)
                        x=(abp.sqrt()*x0+(1-abp).sqrt()*torch.randn_like(x) if t>1 else x0)
                        x=x*(1-mt)+xi*mt
                    samps.append(x.cpu())
            sn=torch.stack(samps,0)[:,0,:,:].numpy()  # (N,NF,L)
            med=np.median(sn[:,:,gidx],axis=0)        # (NF,GL)
            for j,ts in enumerate(fidx[gidx]):
                if msk[0,gidx[j]]==0:
                    long_rows.append({'PtID':pid,'ts':ts,'col':'GlucoseCGM',
                        'value':round(float(med[0,j]*G_STD+G_MEAN),3),
                        'ci_lo':round(float(np.percentile(sn[:,0,gidx[j]],10)*G_STD+G_MEAN),3),
                        'ci_hi':round(float(np.percentile(sn[:,0,gidx[j]],90)*G_STD+G_MEAN),3)})
                if msk[1,gidx[j]]==0:
                    long_rows.append({'PtID':pid,'ts':ts,'col':'HR',
                        'value':round(float(med[1,j]*H_STD+H_MEAN),3),
                        'ci_lo':round(float(np.percentile(sn[:,1,gidx[j]],10)*H_STD+H_MEAN),3),
                        'ci_hi':round(float(np.percentile(sn[:,1,gidx[j]],90)*H_STD+H_MEAN),3)})
            gc.collect()

long_df=pd.DataFrame(long_rows)
print(f"CSDI long gaps imputed: {len(long_df):,} timesteps")


# Final Assembly + Labels + Save

In [ ]:
df_final = df.copy()   

df_final['GlucoseCGM_imputed_final'] = df_final['GlucoseCGM'].copy()
df_final['HR_imputed_final']         = df_final['HR'].copy()

# Base labels from original data
df_final['imputation_method_glucose'] = np.where(
    df_final['GlucoseCGM'].isna(), 'missing', 'real')
df_final['imputation_method_hr'] = np.where(
    df_final['HR'].isna(), 'missing', 'real')

df_final['ts'] = pd.to_datetime(df_final['ts'])

# apply short gap imputations + label 'linear_short' ───
if not short_df.empty:
    short_df['ts'] = pd.to_datetime(short_df['ts'])
    for col, imp_col, method_col in [
        ('GlucoseCGM','GlucoseCGM_imputed_final','imputation_method_glucose'),
        ('HR',        'HR_imputed_final',         'imputation_method_hr'),
    ]:
        s = short_df[short_df['col']==col][['PtID','ts','value']].copy()
        if s.empty: continue
        s = s.drop_duplicates(subset=['PtID','ts'])
        df_final = df_final.merge(
            s.rename(columns={'value': f'_s{col}'}),
            on=['PtID','ts'], how='left')
        # Fill where: short value exists AND original was missing
        sm = df_final[f'_s{col}'].notna() & df_final[imp_col].isna()
        df_final.loc[sm, imp_col]     = df_final.loc[sm, f'_s{col}']
        df_final.loc[sm, method_col] = 'linear_short'
        df_final.drop(columns=[f'_s{col}'], inplace=True)

# apply long gap imputations + label 'csdi_long' ───
if long_df is not None and len(long_df) > 0:
    long_df['ts'] = pd.to_datetime(long_df['ts'])
    for col, imp_col, method_col in [
        ('GlucoseCGM','GlucoseCGM_imputed_final','imputation_method_glucose'),
        ('HR',        'HR_imputed_final',         'imputation_method_hr'),
    ]:
        lg = long_df[long_df['col']==col][['PtID','ts','value']].copy()
        if lg.empty: continue
        lg = lg.drop_duplicates(subset=['PtID','ts'], keep='last')
        df_final = df_final.merge(
            lg.rename(columns={'value': f'_l{col}'}),
            on=['PtID','ts'], how='left')
        # Fill where: long value exists AND still missing after short-gap fill
        lm = df_final[f'_l{col}'].notna() & df_final[imp_col].isna()
        df_final.loc[lm, imp_col]     = df_final.loc[lm, f'_l{col}']
        df_final.loc[lm, method_col] = f'csdi_long'
        df_final.drop(columns=[f'_l{col}'], inplace=True)

# ── Summary ───────────────────────────────────────────────────────
df_final.to_csv(FINAL_FILE, index=False)
print(f"\nSaved: {FINAL_FILE}")
print(f"Rows: {len(df_final):,}")
print(f"\nGlucose label distribution:")
print(df_final['imputation_method_glucose'].value_counts(dropna=False).to_string())
print(f"\nHR label distribution:")
print(df_final['imputation_method_hr'].value_counts(dropna=False).to_string())
g_nan = df_final['GlucoseCGM_imputed_final'].isna().sum()
h_nan = df_final['HR_imputed_final'].isna().sum()
print(f"\nRemaining NaN: Glucose={g_nan:,} | HR={h_nan:,}")
if g_nan > 0:
    print("  (NaN = gaps that could not be imputed due to insufficient context)")
